## Week 1 Day 1 Lab

In [ ]:
# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

## Solution 1: Connecting to Deepseek

In [ ]:
load_dotenv(override=True)
#api_key = os.getenv('OPENAI_API_KEY')
api_key = os.getenv('DEEPSEEK_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk"):
    print("An API key was found, but it doesn't start with sk-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


In [ ]:
deepseek = OpenAI(
    api_key=os.environ.get('DEEPSEEK_API_KEY'),
    base_url="https://api.deepseek.com")

In [ ]:
# Step 1: Create your prompts

system_prompt = """
You are a professional book reviewer and critic that analyzes the synopsis of a book website,
and provides a short really helpful summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block.

"""
user_prompt_pref = """
Suggest a list of 5 children's books that are highly rated and popular, 
along with a brief description of each book based on the website contents. The book should be suitable for children aged 5-10 years old. Please provide the title, author, and a short summary of each book.
"""
website="https://www.amazon.ca/s?k=amazon+children+books&gad_source=1&hvadid=788667809453&hvdev=c&hvexpln=0&hvlocphy=9198282&hvnetw=g&hvocijid=4556621409996784219--&hvqmt=e&hvrand=4556621409996784219&hvtargid=kwd-312629328619&hydadcr=16516_13778724&mcid=a2c5f09d829830caa6f1046b0039ac8d&tag=googcana-20&ref=pd_sl_5fxwvbh2ei_e"

# Step 2: Make the messages list

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_pref + website}
    ]

In [ ]:
# Step 3: Call Deepseek  

def recommend_books(url):
    website = fetch_website_contents(url)
    response = deepseek.chat.completions.create(
    model="deepseek-v4-pro",
    stream=False,
    reasoning_effort="high",
    extra_body={"thinking": {"type": "enabled"}},
    messages=messages_for(website)
    )
    recommendations = response.choices[0].message.content
    return display(Markdown(recommendations))

# Step 4: print the result
recommend_books(website)

## Using the llama3.2 local model to solve the same problem

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

def recommend_books_llama(url):
    website = fetch_website_contents(url)
    response = ollama.chat.completions.create(
    model="llama3.2",
    messages=messages_for(website)
    )
    recommendations = response.choices[0].message.content
    return display(Markdown(recommendations))

# Step 4: print the result
recommend_books_llama(website)